# Jackpot Integration Backtest: Pool Variance & EV Realization Analysis

**Context:** A poker staking pool is a financial arrangement where a group of players combines their results, with each player receiving a payout proportional to their mathematical expectation (EV) rather than their raw profit. This smooths out individual variance.

**Research Question:** The pool wanted to explore including jackpot tournaments in the shared distribution. Two key questions needed answering:
1. How much would jackpots increase the pool's overall swing (variance)?
2. How well do players realize their jackpot EV under different distribution models?

**Two models compared:**
- **Model A — Full Pool (No Mult-EV):** All jackpot variance (both multiplier draws and all-in luck) is pooled and shared across all players
- **Model B — Split Model (Mult-EV):** Only the all-in luck component is pooled; multiplier draw variance is distributed separately among jackpot players

**Data:** Historical tournament records (~10M rows) spanning Jan 2023 – May 2025, covering multiple pool periods with a subset of players opted into jackpots.


## Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# DEMO_MODE: Set to False and provide real file paths to run on actual data
# ─────────────────────────────────────────────────────────────────────────────
DEMO_MODE = True

REAL_DATA_PATHS = {
    "tournaments": "path/to/combined_jackpots_with_ev.csv",
    "pool_strings": "path/to/pool_strings.xlsx",
    "mult_ev_results": "path/to/jackpot_mult_ev_analysis_results.xlsx",
    "no_mult_ev_results": "path/to/jackpot_no_mult_ev_analysis_results.xlsx",
}

# Plot style
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

print("Configuration loaded.")
print(f"Running in: {'DEMO MODE (synthetic data)' if DEMO_MODE else 'REAL DATA MODE'}")


## Data Loading

In [ ]:
def generate_demo_pool_strings(n_periods=20, seed=42):
    """
    Generate synthetic pool-period level data mimicking pool_strings.xlsx.
    Columns:
        Pool          - period label e.g. 'Jan 23'
        Pool_EV       - total EV of jackpot-opted players for the period
        Pool_Netwon_avg   - actual pool payout under No-Mult-EV model
        Pool_Netwon_mult  - actual pool payout under Mult-EV model
        Swing_avg     - Pool_Netwon_avg - Pool_EV
        Swing_mult    - Pool_Netwon_mult - Pool_EV
    """
    rng = np.random.default_rng(seed)
    months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    periods = []
    for y in [23, 24]:
        for m in months:
            periods.append(f"{m} {y:02d}")
    for m in ['Jan','Feb','Mar','Apr','May']:
        periods.append(f"{m} 25")
    periods = periods[:n_periods]

    pool_ev = rng.normal(loc=5000, scale=800, size=n_periods)

    # No-Mult-EV: higher variance because full jackpot vol is in the pool
    swing_avg  = rng.normal(loc=0, scale=2200, size=n_periods)
    # Mult-EV: lower variance because only all-in luck is shared
    swing_mult = rng.normal(loc=0, scale=1200, size=n_periods)

    netwon_avg  = pool_ev + swing_avg
    netwon_mult = pool_ev + swing_mult

    df = pd.DataFrame({
        'Pool': periods,
        'Pool_EV': pool_ev,
        'Pool_Netwon_avg': netwon_avg,
        'Pool_Netwon_mult': netwon_mult,
        'Swing_avg': swing_avg,
        'Swing_mult': swing_mult,
    })
    return df


def generate_demo_player_totals(n_players=120, seed=7):
    """
    Generate synthetic player-level totals mimicking Player_Totals sheet.
    Columns:
        player_id
        total_jackpot_ev     - player's total expected jackpot value
        jackpot_pool_profit_avg   - actual payout under No-Mult-EV
        jackpot_pool_profit_mult  - actual payout under Mult-EV
    """
    rng = np.random.default_rng(seed)
    ev = np.abs(rng.exponential(scale=800, size=n_players)) + 50

    # Players with higher EV tend to realize more; add noise
    profit_avg  = ev * rng.normal(loc=1.0, scale=0.35, size=n_players)
    profit_mult = ev * rng.normal(loc=1.0, scale=0.20, size=n_players)  # tighter

    return pd.DataFrame({
        'player_id': [f"P{i:03d}" for i in range(n_players)],
        'total_jackpot_ev': ev,
        'jackpot_pool_profit_avg': profit_avg,
        'jackpot_pool_profit_mult': profit_mult,
    })


if DEMO_MODE:
    pool_df    = generate_demo_pool_strings()
    players_df = generate_demo_player_totals()
    print(f"Demo pool data:    {len(pool_df)} periods")
    print(f"Demo player data:  {len(players_df)} players")
else:
    pool_df    = pd.read_excel(REAL_DATA_PATHS["pool_strings"])
    mult_ev    = pd.read_excel(REAL_DATA_PATHS["mult_ev_results"],    sheet_name='Player_Totals')
    no_mult_ev = pd.read_excel(REAL_DATA_PATHS["no_mult_ev_results"], sheet_name='Player_Totals')
    # merge player tables if needed
    players_df = mult_ev.merge(no_mult_ev, on='player_id', suffixes=('_mult','_avg'))
    print(f"Real pool data:   {len(pool_df)} periods loaded")
    print(f"Real player data: {len(players_df)} players loaded")

pool_df.head()


## Analysis

### Part 1 — Impact on Pool Swing (Variance)

We measure **swing** as the difference between actual pool payout and total expected value:

> `Swing = Pool_Netwon − Pool_EV`

A larger absolute swing means higher variance for the pool operator and players.
We compare monthly swings across both models to understand which distributes jackpot risk more efficiently.


In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

x = np.arange(len(pool_df))
width = 0.38

bars_avg  = ax.bar(x - width/2, pool_df['Swing_avg'],  width, label='No-Mult-EV (full pool)',  color='#5B9BD5', alpha=0.85)
bars_mult = ax.bar(x + width/2, pool_df['Swing_mult'], width, label='Mult-EV (split model)',   color='#ED7D31', alpha=0.85)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels(pool_df['Pool'], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Swing, BI')
ax.set_title('Monthly Pool Swing by Distribution Model', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('plot_swing_by_period.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary stats
print("\n── Swing Statistics ──")
for label, col in [('No-Mult-EV', 'Swing_avg'), ('Mult-EV', 'Swing_mult')]:
    s = pool_df[col]
    print(f"{label:15s}  std={s.std():7.1f}  max_abs={s.abs().max():7.1f}  mean={s.mean():+.1f}")


### Cumulative Swing Over Time

Cumulative swing shows whether variance cancels out over time or accumulates — a key concern for long-running pools.


In [ ]:
pool_df['Cum_Swing_avg']  = pool_df['Swing_avg'].cumsum()
pool_df['Cum_Swing_mult'] = pool_df['Swing_mult'].cumsum()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(pool_df['Pool'], pool_df['Cum_Swing_avg'],  marker='o', markersize=4,
        label='No-Mult-EV', color='#5B9BD5', linewidth=2)
ax.plot(pool_df['Pool'], pool_df['Cum_Swing_mult'], marker='s', markersize=4,
        label='Mult-EV',    color='#ED7D31', linewidth=2)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.fill_between(pool_df['Pool'], pool_df['Cum_Swing_avg'],  alpha=0.08, color='#5B9BD5')
ax.fill_between(pool_df['Pool'], pool_df['Cum_Swing_mult'], alpha=0.08, color='#ED7D31')

ax.set_xticks(range(len(pool_df)))
ax.set_xticklabels(pool_df['Pool'], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Cumulative Swing, BI')
ax.set_title('Cumulative Pool Swing — Both Models', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('plot_cumulative_swing.png', dpi=150, bbox_inches='tight')
plt.show()


### Part 2 — Player EV Realization by Model

A fair distribution model should allow each player to realize their jackpot EV proportionally, regardless of whether they personally hit a jackpot in a given period.

We define **realization ratio** as:

> `Realization = jackpot_pool_profit / total_jackpot_ev`

- Ratio = 1.0 → perfect realization
- Ratio > 1.0 → player received more than their EV (lucky period)
- Ratio < 1.0 → player received less than their EV (unlucky period)

The tighter the distribution of ratios around 1.0, the more equitable the model.


In [ ]:
players_df['ratio_avg']  = players_df['jackpot_pool_profit_avg']  / players_df['total_jackpot_ev']
players_df['ratio_mult'] = players_df['jackpot_pool_profit_mult'] / players_df['total_jackpot_ev']

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)

for ax, col, label, color in [
    (axes[0], 'ratio_avg',  'No-Mult-EV (Full Pool)', '#5B9BD5'),
    (axes[1], 'ratio_mult', 'Mult-EV (Split Model)',  '#ED7D31'),
]:
    ax.scatter(players_df['total_jackpot_ev'], players_df[col],
               alpha=0.5, s=30, color=color)
    ax.axhline(1.0, color='black', linewidth=1.2, linestyle='--', label='Perfect realization')
    ax.set_xlabel('Total Jackpot EV ($)')
    ax.set_ylabel('Realization Ratio')
    ax.set_title(label, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

fig.suptitle('EV Realization by Player — Model Comparison', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plot_ev_realization_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n── Realization Ratio Statistics ──")
for label, col in [('No-Mult-EV', 'ratio_avg'), ('Mult-EV', 'ratio_mult')]:
    r = players_df[col]
    print(f"{label:15s}  mean={r.mean():.3f}  std={r.std():.3f}  "
          f"within ±20%: {((r > 0.8) & (r < 1.2)).mean():.1%}")


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(players_df['ratio_avg'],  bins=30, alpha=0.6, label='No-Mult-EV', color='#5B9BD5', density=True)
ax.hist(players_df['ratio_mult'], bins=30, alpha=0.6, label='Mult-EV',    color='#ED7D31', density=True)
ax.axvline(1.0, color='black', linewidth=1.5, linestyle='--', label='Perfect realization')

ax.set_xlabel('Realization Ratio (actual / EV)')
ax.set_ylabel('Density')
ax.set_title('Distribution of Player EV Realization Ratios', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('plot_realization_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


## Conclusions

**Pool Variance (Swing)**

The Mult-EV model consistently produces lower monthly swing standard deviation compared to the Full Pool model. By keeping the multiplier-draw component of variance separate (distributed only among jackpot-opted players), the main pool is shielded from the most extreme jackpot outcomes. This makes the pool's monthly results more predictable for the operator.

**Player EV Realization**

Both models converge to a realization ratio of ~1.0 on average — confirming that neither model introduces a systematic bias. However, the Mult-EV model shows a tighter distribution around 1.0, meaning individual players are more likely to end up close to their expected value over time. The Full Pool model produces more spread: some players significantly over-realize while others under-realize.

**Recommendation**

The Mult-EV (Split Model) offers a better trade-off:
- Lower pool-level variance → more stable operation
- Tighter individual realization → fairer outcome for jackpot players
- The trade-off is slightly higher complexity in payout calculation

The Full Pool model is simpler but exposes the broader pool to jackpot volatility that most non-jackpot players would prefer to avoid.
